## Tools

#### Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairing of:

#### 1. Schema, including the name of the tool, a description, and/or argument definitions(often a JSON schema)
#### 2. A function or coroutine to execute. 


In [2]:
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:openai/gpt-oss-120b")

response = model.invoke("Color of Parrot")
response

AIMessage(content='Parrots are famous for their vivid, rainbow‑like plumage, but the exact colors you’ll see depend on the species (and sometimes the individual’s age or sex). Here’s a quick guide to the most common color patterns you’ll encounter:\n\n| Species (or Group) | Typical Color(s) | Notable Variations |\n|--------------------|------------------|--------------------|\n| **Budgerigar (Budgie)** | Bright green (wild type) with yellow face; many captive “morphs” in blue, white, yellow, opaline, pied, etc. | Albino, lutino, pied, cinnamon, etc. |\n| **African Grey** | Mostly medium‑gray body, white face mask, bright red tail feather (the “tail‑band”). | No major color morphs in the wild; captive “white‑faced” morph exists. |\n| **Macaws** (e.g., Blue‑and‑Gold, Scarlet, Green‑Winged) | Bold blocks of blue, gold, red, green, and yellow. | Some species have mostly one dominant hue (e.g., Blue‑and‑Gold = blue & yellow). |\n| **Cockatiels** | Light gray body, orange/red cheek patches, 

In [3]:
from langchain.tools import tool
@tool
def get_weather(location: str) -> str:
    """Get the weather at a location"""
    return f"It's a sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [7]:
response = model_with_tools.invoke("What is the whether in Varanasi right now ?")
print(response)

for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
  

content='' additional_kwargs={'reasoning_content': 'The user asks: "What is the whether in Varanasi right now ?" Probably typo "weather". Need to get current weather for Varanasi using function get_weather. Use function.', 'tool_calls': [{'id': 'fc_c9877b4c-2ba9-4f1b-a8da-6dbcf4079b80', 'function': {'arguments': '{"location":"Varanasi"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 131, 'total_tokens': 199, 'completion_time': 0.141638541, 'completion_tokens_details': {'reasoning_tokens': 39}, 'prompt_time': 0.00527059, 'prompt_tokens_details': None, 'queue_time': 0.39339101, 'total_time': 0.146909131}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e1a78f200e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a196aa-528e-74c3-8ba7-3765d9ee920a-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Varanasi'}, 'id': 'fc

## Tool Execution Loops

In [13]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What is the weather in Varanasi?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)




The current weather in Varanasi is sunny. Enjoy the clear skies!


In [14]:
messages


[{'role': 'user', 'content': 'What is the weather in Varanasi?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks for weather in Varaniki. Need to fetch via get_weather function.', 'tool_calls': [{'id': 'fc_3b030b2e-f263-4a87-8794-ee4e34c51f48', 'function': {'arguments': '{"location":"Varanasi"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 129, 'total_tokens': 176, 'completion_time': 0.098017474, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.005040825, 'prompt_tokens_details': None, 'queue_time': 0.213043796, 'total_time': 0.103058299}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a655021f7d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a196b5-6b22-77c9-98d0-58b8824ccbc0-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Varanasi'}, 'id': 'fc_3b030b2e-f